# Figure 3 and Figures S3B-C, S7, S8, S10-S12: classifier performance and LASSO features

Evaluates the Monte Carlo cross-validated LASSO classifiers (101 train/test splits) and plots
their test AUCs, calibration, selected transcripts, clinical comparisons and the Cohort 3 holdout.

**Inputs**
- `input_dir`: `metadata/sample_metadata.tsv`, `reference/gencode.biotype.name.key.tsv.gz`,
  `counts/counts_norm_cart_paper.tsv.gz` (SFig10C, SFig12B)
- `results/ml/` (from `code/01_compute`):
  - `all/{LN,FMAC,TEX,LASSO,LASSO_SHUFFLE,RAND_30genes}/`: repeated CV, all 91 patients
  - `dlbcl/`: repeated CV, de novo DLBCL only (SFig3B-C)
  - `clinical_vars/`, `forced_clin_ridge/`: clinical-variable and LN + clinical models (SFig10A-B)
  - `holdout_c3/`: train on Cohorts 1-2, test on Cohort 3, bootstrap CI and label permutations (SFig8)

**Outputs** (`plots/`)
- `Fig3A_auc_violins.pdf`, `Fig3AB_ml_results.pdf`
- `SFig3B_auc_full_vs_dlbcl.pdf`, `SFig3C_lasso_features_dlbcl.pdf`
- `SFig7A_auc_empirical_ci.pdf`, `SFig7B_calibration.pdf`
- `SFig8_cv_vs_cohort3_holdout.pdf`
- `SFig10A_auc_forest_genesets_clinical.pdf`, `SFig10B_auc_forest_ln_plus_clinical.pdf`, `SFig10C_decision_curve.pdf`
- `SFig11_lasso_coselection.pdf`
- `SFig12A_selection_real_labels.pdf`, `SFig12A_selection_shuffled_labels.pdf`, `SFig12B_spearman_heatmap.pdf`

In [ ]:
source("../../utils/setup.R")
suppressPackageStartupMessages({
  library(pheatmap)
  library(RColorBrewer)
  library(dcurves)
})

## Inputs

In [ ]:
meta     <- load_sample_metadata()   # 91 patients, response NR/R
gene_key <- load_gene_key()

ML_ALL     <- res_in("ml", "all")
ML_DLBCL   <- res_in("ml", "dlbcl")
ML_CLIN    <- res_in("ml", "clinical_vars")
ML_LN_CLIN <- res_in("ml", "forced_clin_ridge")
ML_HOLDOUT <- res_in("ml", "holdout_c3")

# LASSO selected features (gene_name added from the GENCODE key)
feats_all     <- read_lasso_features(file.path(ML_ALL, "LASSO", "selected_features.csv"), gene_key)
feats_dlbcl   <- read_lasso_features(file.path(ML_DLBCL, "LASSO", "selected_features.csv"), gene_key)
feats_shuffle <- read_lasso_features(file.path(ML_ALL, "LASSO_SHUFFLE", "selected_features.csv"), gene_key)
n_iter <- max(feats_all$iteration) + 1

# Samples scored by the all-transcript models (SFig12B)
lasso_samples <- unique(read.csv(file.path(ML_ALL, "LASSO", "classifier_scores.csv"))$sample_id)

# Cohort 3 holdout (SFig8)
HOLDOUT_METHODS <- c("FMAC", "TEX", "LN", "LASSO")
holdout_summary <- bind_rows(lapply(HOLDOUT_METHODS, function(m)
  read.csv(file.path(ML_HOLDOUT, m, "auc_summary.csv")) %>% mutate(method = NAME_MAP[[m]])))
holdout_boot <- bind_rows(lapply(HOLDOUT_METHODS, function(m)
  read.csv(file.path(ML_HOLDOUT, m, "bootstrap_aucs.csv")) %>% mutate(method = NAME_MAP[[m]])))
holdout_perm <- read.csv(file.path(ML_HOLDOUT, "permutation_aucs.csv"))

# Normalized counts (SFig10C decision curve, SFig12B correlations)
counts_norm <- load_counts("norm")

## Test AUCs of the repeated cross-validation

Train and test AUC of every iteration for the six feature sets (all patients). These AUCs feed
Fig3A, SFig3B, SFig7A, SFig8 (CV part) and SFig10A-B.

In [ ]:
CV_METHODS <- c("FMAC", "TEX", "LN", "LASSO", "RAND_30genes", "LASSO_SHUFFLE")
auc_all <- roc_by_iteration(ML_ALL, CV_METHODS, meta)   # AUC in percent

auc_all %>% group_by(method) %>% summarise(median_test = median(test), .groups = "drop")

## Figure 3A — Test AUC over 101 CV iterations, by feature set

In [ ]:
VIOLIN_LEVELS <- c("All Transcripts", "LN", "FMAC", "TEX", "Random Genes")

violin_df <- auc_all %>%
  mutate(test = test / 100, method = NAME_MAP[method]) %>%
  filter(method %in% VIOLIN_LEVELS) %>%
  mutate(method = factor(method, levels = VIOLIN_LEVELS))

# Pairs shown (BH-adjusted Wilcoxon, non-significant hidden): AT-LN, AT-TEX, LN-FMAC, LN-TEX,
# LN-Random, FMAC-TEX (elements 1, 3, 5, 6, 7, 8 of all pairs)
comparisons <- combn(VIOLIN_LEVELS, 2, simplify = FALSE)[c(1, 3, 5, 6, 7, 8)]
SEP <- 0.03
START <- 0.95

violin_plt <- ggplot(violin_df, aes(x = method, y = test, fill = method, color = method)) +
  geom_violin(width = 0.8, alpha = 0.3, linewidth = 0.4) +
  stat_summary(fun = median, geom = "point", size = 1.2, fill = "red", stroke = NA, alpha = 0.5) +
  scale_fill_manual(values = METHOD_COLORS_VIOLIN) +
  scale_color_manual(values = METHOD_COLORS_VIOLIN) +
  labs(x = NULL, y = "Test AUC") +
  theme_cfrna_print() +
  theme(panel.grid.major.y = element_blank()) +
  theme_axis_lines() +
  stat_compare_means(comparisons = comparisons, method = "wilcox.test", method.args = WILCOX_ARGS,
                     p.adjust.method = "BH",
                     label = "p.signif", hide.ns = TRUE,
                     label.y = START + SEP * 0:5, vjust = 0.65, size = 2, tip.length = 0)

# Pairwise Wilcoxon p-values behind the stars
bind_rows(lapply(comparisons, function(cmp) {
  d <- filter(violin_df, method %in% cmp)
  data.frame(group1 = cmp[1], group2 = cmp[2],
             p = wilcox.test(test ~ droplevels(method), data = d, exact = FALSE)$p.value)
}))

save_plot(violin_plt, "Fig3A_auc_violins", width = 2.75, height = 1.5)
violin_plt

## Figure 3B — Top 10 LASSO transcripts: selection frequency and coefficients

Saved together with Fig3A as `Fig3AB_ml_results.pdf`.

In [ ]:
gene_bar   <- lasso_selection_barplot(feats_all, top_n = 10, title = "LASSO Transcript Selection")
gene_order <- attr(gene_bar, "gene_order")
p_box      <- lasso_coef_boxplot(feats_all, gene_order)

lasso_selection_table(feats_all) %>% head(10)

# Bars and coefficient boxes share the y axis: drop the inner margins and the box y labels.
lasso_panels <- function(gene_bar, p_box, gene_order) {
  p1 <- gene_bar + scale_y_discrete(limits = rev(gene_order)) +
    theme(plot.margin = margin(0, -10, 0, 0)) +
    theme_axis_lines() + theme(panel.grid.major.y = element_line())
  p2 <- p_box + scale_y_discrete(limits = rev(gene_order)) +
    theme_axis_lines(axis = "x") + theme(panel.grid.major.y = element_line()) +
    theme(plot.margin = margin(0, 0, 0, -10), axis.text.y = element_blank(),
          axis.ticks.y = element_blank())
  list(p1, p2)
}
fig3b <- lasso_panels(gene_bar, p_box, gene_order)

fig3ab <- (violin_plt + theme(panel.grid.major.x = element_line())) + plot_spacer() +
  fig3b[[1]] + fig3b[[2]] + plot_layout(widths = c(2.5, 0.1, 1.8, 2.2))
save_plot(fig3ab, "Fig3AB_ml_results", width = 6.8, height = 2)
fig3ab

## Figure S3B — Test AUC, all patients vs de novo DLBCL only

In [ ]:
SENS_METHODS <- c("FMAC", "TEX", "LN", "LASSO")
auc_dlbcl <- roc_by_iteration(ML_DLBCL, SENS_METHODS, meta)

dlbcl_cmp_df <- bind_rows(
  auc_all %>% filter(method %in% SENS_METHODS) %>% mutate(cohort = "All patients"),
  auc_dlbcl %>% mutate(cohort = "De novo DLBCL only")
) %>%
  mutate(method = factor(NAME_MAP[method], levels = c("TEX", "FMAC", "LN", "All Transcripts")),
         cohort = factor(cohort, levels = c("All patients", "De novo DLBCL only")),
         test   = test / 100)

dlbcl_cmp_df %>% group_by(method, cohort) %>% summarise(median_test = median(test), .groups = "drop")

dlbcl_colors <- setNames(SENSITIVITY_COLORS, c("All patients", "De novo DLBCL only"))

p_dlbcl <- ggplot(dlbcl_cmp_df, aes(x = cohort, y = test, fill = cohort, color = cohort)) +
  geom_violin(width = 0.8, alpha = 0.7, linewidth = 0.4) +
  stat_summary(fun = median, geom = "point", size = 2.5,
               fill = "black", color = "black", stroke = NA, alpha = 1.0) +
  stat_compare_means(comparisons = list(c("All patients", "De novo DLBCL only")),
                     method = "wilcox.test", method.args = WILCOX_ARGS, label = "p.signif",
                     hide.ns = FALSE, size = 2.5, tip.length = 0) +
  facet_wrap(~method, nrow = 1) +
  scale_fill_manual(values = dlbcl_colors) +
  scale_color_manual(values = dlbcl_colors) +
  labs(x = NULL, y = "Test AUC", title = "Sensitivity analysis: de novo DLBCL only") +
  theme_classic(base_size = 9) +
  theme(legend.position  = "bottom",
        legend.text      = element_text(size = 10),
        legend.title     = element_blank(),
        strip.background = element_blank(),
        strip.text       = element_text(size = 10),
        axis.text.y      = element_text(size = 10),
        axis.text.x      = element_blank(),
        axis.ticks.x     = element_blank())

save_plot(p_dlbcl, "SFig3B_auc_full_vs_dlbcl", width = 5, height = 3)
p_dlbcl

## Figure S3C — Top 10 LASSO transcripts, de novo DLBCL models

In [ ]:
bar_dlbcl   <- lasso_selection_barplot(feats_dlbcl, top_n = 10, title = "LASSO Selection")
order_dlbcl <- attr(bar_dlbcl, "gene_order")
sfig3c <- lasso_panels(bar_dlbcl, lasso_coef_boxplot(feats_dlbcl, order_dlbcl), order_dlbcl)

lasso_selection_table(feats_dlbcl) %>% head(10)
cat("Shared with the all-patient top 10:", paste(intersect(order_dlbcl, gene_order), collapse = ", "), "\n")

p_sfig3c <- sfig3c[[1]] + sfig3c[[2]] + plot_layout(widths = c(1.8, 2.2))
save_plot(p_sfig3c, "SFig3C_lasso_features_dlbcl", width = 6.8 / 2, height = 2)
p_sfig3c

## Figure S7A — Median test AUC with 95% empirical interval

In [ ]:
FOUR_LEVELS <- c("TEX", "FMAC", "LN", "All Transcripts")

ci_cv4 <- auc_all %>%
  filter(method %in% SENS_METHODS) %>%
  mutate(test = test / 100, method = NAME_MAP[method]) %>%
  auc_ci_summary(by = "method")
ci_cv4

p_sfig7a <- forest_plot(ci_cv4 %>% mutate(method = factor(method, levels = FOUR_LEVELS)),
                        colors = METHOD_COLORS, x_lab = "Test AUC (95% empirical CI)",
                        point_size = 0.4, x_breaks = NULL, small_axis_title = FALSE)
save_plot(p_sfig7a, "SFig7A_auc_empirical_ci", width = 2, height = 2)
p_sfig7a

## Figure S7B — Calibration of the pooled test-set predictions

All test-set predictions of the 101 iterations are pooled per model and binned into deciles of
predicted probability. The Brier scores are quoted in the text (LN 0.223, FMAC 0.239, TEX 0.284).

In [ ]:
cal_scores <- bind_rows(lapply(SENS_METHODS, function(m) {
  read.csv(file.path(ML_ALL, m, "classifier_scores.csv")) %>%
    filter(train_or_test == "test") %>%
    left_join(meta[, c("sample_id", "response")], by = "sample_id") %>%
    mutate(y = ifelse(response == "R", 1L, 0L), method = NAME_MAP[[m]])
}))

prevalence <- mean(cal_scores$y[cal_scores$method == "All Transcripts"])
cat(sprintf("Null model Brier score (predict prevalence): %.3f\n", prevalence * (1 - prevalence)))
cal_scores %>%
  group_by(method) %>%
  summarise(brier = mean((classifier_score - y)^2), n = n(), .groups = "drop") %>%
  arrange(brier)

cal_plot_df <- cal_scores %>%
  mutate(bin = cut(classifier_score, breaks = seq(0, 1, by = 0.1), include.lowest = TRUE)) %>%
  group_by(method, bin) %>%
  summarise(mean_pred = mean(classifier_score), obs_freq = mean(y), n = n(), .groups = "drop") %>%
  mutate(method = factor(method, levels = rev(FOUR_LEVELS)))

p_sfig7b <- ggplot(cal_plot_df, aes(x = mean_pred, y = obs_freq, color = method)) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey50", linewidth = 0.4) +
  geom_line(linewidth = 0.6, alpha = 0.8) +
  geom_point(aes(size = n), alpha = 0.8) +
  scale_color_manual(values = METHOD_COLORS) +
  scale_size_continuous(range = c(1, 3), guide = "none") +
  scale_x_continuous(limits = c(0, 1)) +
  scale_y_continuous(limits = c(0, 1)) +
  labs(x = "Mean predicted probability", y = "Observed response rate") +
  theme_cfrna_print() +
  theme_axis_lines()
save_plot(p_sfig7b, "SFig7B_calibration", width = 2, height = 2)
p_sfig7b

## Test AUCs of the gene-set, clinical and LN + clinical models (SFig8, SFig10A-B)

Test AUC (0-1) per iteration. The gene-set models reuse the AUCs above; the clinical-variable
models (`clinical_vars/`) and the LN + clinical ridge models (`forced_clin_ridge/`) are computed here.

In [ ]:
CLIN_METHODS    <- c("bridging", "IPI", "LDH", "subtype", "histology", "stage")
LN_CLIN_METHODS <- c("LN_bridging", "LN_IPI", "LN_LDH", "LN_subtype", "LN_histology",
                     "LN_stage", "LN_clinical_all")

aucs_from_dir <- function(dir, methods) {
  bind_rows(lapply(methods, function(m)
    data.frame(method = m, test = compute_aucs(file.path(dir, m, "classifier_scores.csv"), meta))))
}

auc_models <- bind_rows(
  auc_all %>% transmute(method, test = test / 100),
  aucs_from_dir(ML_CLIN, CLIN_METHODS),
  aucs_from_dir(ML_LN_CLIN, LN_CLIN_METHODS)
) %>%
  mutate(method = unname(NAME_MAP[method]), group = unname(GROUP_MAP[method]))

ci_models <- auc_ci_summary(auc_models) %>% arrange(group, median_auc)
print(ci_models, n = Inf)

## Figure S8 — Repeated cross-validation vs Cohort 3 holdout

Left: median test AUC and 95% empirical interval over the 101 CV iterations (all patients). Right: models trained on Cohorts 1-2 and tested
on Cohort 3, with the 95% bootstrap interval of the test AUC.

In [ ]:
SFIG8_LEVELS <- c("TEX", "All Transcripts", "FMAC", "LN")

holdout_summary %>%
  select(method, train_auc, test_auc, ci_low, ci_high) %>%
  mutate(across(where(is.numeric), ~ round(.x, 3)))

# Bootstrap distributions behind the intervals
holdout_boot %>%
  group_by(method) %>%
  summarise(n_boot = n(), median = median(bootstrap_auc),
            q2.5 = quantile(bootstrap_auc, 0.025), q97.5 = quantile(bootstrap_auc, 0.975),
            .groups = "drop")

# Label permutations: empirical p = (1 + #perm AUC >= observed) / (1 + n_perm)
perm_obs <- setNames(holdout_summary$test_auc, holdout_summary$method)
bind_rows(lapply(c(LN = "LN", LASSO = "All Transcripts"), function(lbl) {
  m <- names(NAME_MAP)[match(lbl, NAME_MAP)]
  perm <- holdout_perm[[paste0(m, "_test_auc")]]
  data.frame(method = lbl, observed = perm_obs[[lbl]], n_perm = length(perm),
             perm_median = median(perm), p_perm = (1 + sum(perm >= perm_obs[[lbl]])) / (1 + length(perm)))
}))

p_cv <- forest_plot(ci_models %>% filter(method %in% SFIG8_LEVELS) %>%
                      mutate(method = factor(method, levels = SFIG8_LEVELS)),
                    x_lab = "Test AUC (95% CI)", y_lab = "Gene Set") +
  ggtitle("Repeated CV (101 iterations)")
p_holdout <- forest_plot(holdout_summary %>% mutate(method = factor(method, levels = SFIG8_LEVELS)),
                         x = "test_auc", x_lab = "Test AUC (95% bootstrap CI)") +
  ggtitle("Cohort 3 holdout") +
  theme(axis.text.y = element_blank(), axis.ticks.y = element_blank())

p_sfig8 <- (p_cv + p_holdout) & theme(plot.title = element_text(size = 7, hjust = 0.5))
save_plot(p_sfig8, "SFig8_cv_vs_cohort3_holdout", width = 5, height = 2)
p_sfig8

## Figure S10A — Test AUC of gene-set and single clinical-variable models

In [ ]:
# Rows: controls (bottom), clinical variables, gene sets (top); each by ascending median AUC
univ_df <- ci_models %>% filter(group %in% c("Control", "Clinical", "Gene set"))
univ_order <- unlist(lapply(c("Control", "Clinical", "Gene set"), function(g)
  univ_df %>% filter(group == g) %>% arrange(median_auc) %>% pull(method)))

p_sfig10a <- forest_plot(univ_df %>% mutate(method = factor(method, levels = univ_order)),
                         x_lab = "Test AUC", y_lab = "Variable")
save_plot(p_sfig10a, "SFig10A_auc_forest_genesets_clinical", width = 3.0, height = 3.5)
p_sfig10a

## Figure S10B — Test AUC of LN combined with clinical variables

In [ ]:
ln_clin_df <- ci_models %>% filter(group == "Gene Set + Clinical" | method == "LN")
ln_clin_order <- c(ln_clin_df %>% filter(group == "Gene Set + Clinical") %>% arrange(median_auc) %>% pull(method),
                   "LN")

p_sfig10b <- forest_plot(ln_clin_df %>% mutate(method = factor(method, levels = ln_clin_order)),
                         x_lab = "Test AUC", y_lab = "Variable")
save_plot(p_sfig10b, "SFig10B_auc_forest_ln_plus_clinical", width = 3.0, height = 3.5)
p_sfig10b

## Figure S10C — Decision curve: LN score, clinical model and both

LN score (`AddGeneSetScore`, nbin = 25) on the normalized counts collapsed to gene symbols
(version-stripped Ensembl IDs, duplicate symbols averaged) and restricted to genes with value
> 0.25 in at least 25% of samples. Logistic models of response (1 = R): clinical (6 variables,
complete cases), LN score (fitted on all 91 patients) and clinical + LN. The decision curves are
computed on the complete cases.

In [ ]:
counts_sym <- ensembl_to_symbol(counts_norm, gene_key, strip_version = TRUE, duplicates = "mean")
detected   <- rowSums(counts_sym > 0.25) >= 0.25 * ncol(counts_sym)
counts_sym <- counts_sym[detected, ]
cat("Genes detected in >= 25% of samples:", nrow(counts_sym), "\n")

meta_dca <- recode_clinical(meta, "dca")
DCA_CLIN_VARS <- c("LDH_Day30", "LDH_prelymph", "bridging_therapy", "stage", "IPI", "histology")

ln_score <- AddGeneSetScore(as.matrix(counts_sym[, meta_dca$sample_id]), LN_SYMBOLS,
                            nbin = 25, name = "LN_set")
dca_df <- meta_dca %>%
  mutate(LN_set1 = ln_score[sample_id, "LN_set1"], response_bin = ifelse(response == "R", 1, 0))
dca_complete <- dca_df %>% filter(complete.cases(dca_df[, DCA_CLIN_VARS]))
cat("All patients:", nrow(dca_df), "| complete clinical cases:", nrow(dca_complete), "\n")
print(table(dca_complete$response))

fit_clinical <- glm(reformulate(DCA_CLIN_VARS, "response_bin"), data = dca_complete, family = binomial())
fit_ln       <- glm(response_bin ~ LN_set1, data = dca_df, family = binomial())
fit_combined <- glm(reformulate(c(DCA_CLIN_VARS, "LN_set1"), "response_bin"),
                    data = dca_complete, family = binomial())
summary(fit_ln)$coefficients
summary(fit_combined)$coefficients

dca_complete <- dca_complete %>%
  mutate(Clinical_model = predict(fit_clinical, newdata = ., type = "response"),
         LN_model       = predict(fit_ln, newdata = ., type = "response"),
         Combined_model = predict(fit_combined, newdata = ., type = "response"))

dca_res <- dca(response_bin ~ Clinical_model + LN_model + Combined_model,
               data = dca_complete, thresholds = seq(0, 1, by = 0.01),
               label = list(Clinical_model = "Clinical", LN_model = "LN score",
                            Combined_model = "Clinical + LN"))
p_sfig10c <- plot(dca_res, smooth = TRUE)
save_plot(p_sfig10c, "SFig10C_decision_curve", width = 5, height = 4)
p_sfig10c

## Figure S11 — Co-selection frequency of the top 10 LASSO transcripts

In [ ]:
cosel_mat <- coselection_matrix(feats_all, top_n = 10)
round(cosel_mat, 2)

p_sfig11 <- coselection_heatmap(cosel_mat, n_iter = n_iter)
save_plot(p_sfig11, "SFig11_lasso_coselection", width = 4.5, height = 4)
p_sfig11

## Figure S12A — Selection frequency with real vs shuffled labels

In [ ]:
p_sel_real    <- lasso_selection_barplot(feats_all, top_n = 10, title = "LASSO Transcript Selection")
p_sel_shuffle <- lasso_selection_barplot(feats_shuffle, top_n = 10, title = "LASSO Shuffle Transcript Selection")
lasso_selection_table(feats_shuffle) %>% head(10)

save_plot(p_sel_real, "SFig12A_selection_real_labels", width = 3, height = 3)
save_plot(p_sel_shuffle, "SFig12A_selection_shuffled_labels", width = 3, height = 3)
p_sel_real
p_sel_shuffle

## Figure S12B — Spearman correlation of the top 10 LASSO transcripts

Normalized counts of the samples used by the all-transcript models. The co-selection frequency of
each gene pair (SFig11) is compared with its expression correlation.

In [ ]:
id_map <- symbol_to_ensembl(rownames(cosel_mat), gene_key, rownames(counts_norm))
stopifnot(all(lasso_samples %in% colnames(counts_norm)))

expr_sub <- t(counts_norm[id_map$gene_id, lasso_samples, drop = FALSE])
colnames(expr_sub) <- id_map$gene_name[match(colnames(expr_sub), id_map$gene_id)]
gene_corr <- cor(expr_sub, method = "spearman", use = "pairwise.complete.obs")

# Rows use the reversed dendrogram (dendrograms hidden)
hc_expr <- hclust(as.dist(1 - gene_corr))
p_sfig12b <- pheatmap(gene_corr,
                      color           = colorRampPalette(rev(brewer.pal(9, "RdBu")))(100),
                      breaks          = seq(-1, 1, length.out = 101),
                      border_color    = "white",
                      cluster_rows    = as.hclust(rev(as.dendrogram(hc_expr))),
                      cluster_cols    = hc_expr,
                      treeheight_row  = 0,
                      treeheight_col  = 0,
                      angle_col       = 45,
                      fontsize        = 7,
                      fontsize_number = 5,
                      display_numbers = TRUE,
                      number_format   = "%.2f",
                      number_color    = "black",
                      silent          = TRUE)
save_plot(p_sfig12b, "SFig12B_spearman_heatmap", width = 4.5, height = 4)

pair_df <- bind_rows(lapply(combn(id_map$gene_name, 2, simplify = FALSE), function(p)
  data.frame(gene1 = p[1], gene2 = p[2],
             coselection_freq = cosel_mat[p[1], p[2]], expr_corr = gene_corr[p[1], p[2]])))
cor_test <- suppressWarnings(cor.test(pair_df$expr_corr, pair_df$coselection_freq, method = "spearman"))
cat(sprintf("Co-selection frequency vs. expression correlation: rho = %.3f, p = %.4g (n = %d pairs)\n",
            cor_test$estimate, cor_test$p.value, nrow(pair_df)))

grid::grid.draw(p_sfig12b$gtable)